# CLIP-ViT + LoRA Continual Learning on Split CIFAR-100

This notebook implements a compact, reproducible comparison setup.

Main setup:

- CLIP-ViT vision encoder: `openai/clip-vit-base-patch16`
- Split CIFAR-100 continual learning
- 5 steps, 20 classes per step
.

Main comparison:

1. `seq_ft_no_replay`
2. `simple_avg_no_replay`
3. `simple_avg_replay`
4. `do_merging_simple`
5. `orthogonal_loss`
6. `rank_extension`
7. `joint_upper_bound`


In [ ]:
# ============================================================
# 1) Imports
# ============================================================

import os
import gc
import json
import random
import math
import inspect
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset, concatenate_datasets
from torchvision import transforms

from transformers import (
    CLIPImageProcessor,
    CLIPVisionModel,
    TrainingArguments,
    Trainer,
    set_seed,
)

from transformers.modeling_outputs import ImageClassifierOutput

from peft import LoraConfig, get_peft_model

try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

In [ ]:
# ============================================================
# 2) Full comparison configuration
# ============================================================

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEBUG_MODE = False
FAST_RUN = False

RUN_NAME_BASE = "clip_vit_lora_cifar100_full_comparison_with_orth_rankext"
RUN_NAME = f"{RUN_NAME_BASE}_{'FAST_RUN_DEBUG' if FAST_RUN else 'EPOCH3_MAIN'}"

MODEL_CHECKPOINT = "openai/clip-vit-base-patch16"

NUM_CLASSES = 100
NUM_STEPS = 5
CLASSES_PER_STEP = 20

# ============================================================
# Epoch setup (forced fast for debug pass)
# ============================================================
FULL_FT_EPOCHS = 3
FULL_LORA_EPOCHS = 3
FULL_JOINT_EPOCHS = 3
FULL_ORTH_EPOCHS = 3
FULL_RANKEXT_EPOCHS = 3

SCRATCH_EPOCHS = 3

FT_EPOCHS = 3
LORA_EPOCHS = 3
JOINT_EPOCHS = 3
ORTH_EPOCHS = 3
RANKEXT_EPOCHS = 3

# ============================================================
# Batch settings
# ============================================================

BATCH_FT = 8
ACCUM_FT = 2

BATCH_LORA = 16
ACCUM_LORA = 1

# ============================================================
# Learning rates
# ============================================================

LR_FT = 3e-5
LR_LORA = 5e-5
LR_JOINT = 5e-5

LR_ORTH = 5e-5
LR_RANKEXT = 1e-4

WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.10
SCHED = "cosine"

USE_FP16 = torch.cuda.is_available()

# ============================================================
# LoRA setup for CLIP-ViT
# ============================================================

LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj"]

# ============================================================
# Replay setup (kept for compatibility, disabled in active methods)
# ============================================================

REPLAY_PER_CLASS = 20
RANKEXT_REPLAY_PER_CLASS = REPLAY_PER_CLASS

# ============================================================
# Orthogonal-loss setup (fixed lambdas, no auto-ratio)
# ============================================================

LAMBDA_ORTH = 1.0
ORTH_LOSS_TYPE = "trace"
ORTH_SCALE_MODE = "squared_trace"
ORTH_TARGET_RATIO = 0.0
ORTH_LAMBDA_MIN = 1e-6
ORTH_LAMBDA_MAX = 1e3
ORTH_EPS = 1e-12
ORTH_LOSS_LOG_EVERY = 1
USE_IPC_CONSTRAINT = False
LAMBDA_IPC = 0.0
IPC_TOP_P = 0.10
IPC_IMPORTANCE_NUM_BATCHES = 8

LAMBDA_ORTH_TRACE_LIST = [1.0]
LAMBDA_ORTH_NORM_LIST = [500.0]
ORTH_NORM_EPS = 1e-12
LAMBDA_ORTH_FACTOR_LIST = [0.5, 1.0, 10.0, 50.0]

ORTH_CONFIG_SWEEP = []
ENABLE_RANKEXT_ORTH_CONFIG_SWEEP = False
ORTH_DIAGNOSTICS = True
RANKEXT_DIAGNOSTICS = True

# ============================================================
# Rank-extension setup
# ============================================================

RANKEXT_NEW_RANK_PER_STEP = 8
RANKEXT_ALPHA_PER_RANK = 2.0

# ============================================================
# Active methods
# Keep legacy keys for compatibility, but disable unused methods.
# ============================================================

METHODS_TO_RUN = {
    # focused methods
    "simple_avg": True,
    "rank_extension": True,
    "rank_extension_orth_factor_lam_0p5": True,
    "rank_extension_orth_factor_lam_1": True,
    "rank_extension_orth_factor_lam_10": True,
    "rank_extension_orth_factor_lam_50": True,

    # disabled methods
    "full_finetune": False,
    "seq_ft_no_replay": False,
    "simple_avg_no_replay": False,
    "simple_avg_replay": False,
    "simple_avg_orth": False,
    "do_merging_simple": False,
    "do_merging_simple_orth": False,
    "orthogonal_loss": False,
    "rank_extension_replay": False,
    "rank_extension_orth": False,
    "rank_extension_replay_orth": False,
    "rank_extension_zero_old_merge": False,
    "rank_extension_orth_trace": False,
    "rank_extension_orth_norm": False,
    "rank_extension_orth_trace_abs_lam_1": False,
    "rank_extension_orth_norm_lam_500": False,
    "rank_extension_orth_factor_lam_0p1": False,
    "rank_extension_zero_old_merge_orth_trace": False,
    "rank_extension_zero_old_merge_orth_norm": False,
    "joint_upper_bound": False,
}

# ============================================================
# Fixed output path on cluster
# ============================================================

ROOT_RESULTS_DIR = "/nfsd/lttm4/tesisti/shahrampour/projects/runs/cifar100_5step_n5_main/results"

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")

BASE_OUTPUT_DIR = os.path.join(
    ROOT_RESULTS_DIR,
    f"{RUN_NAME}_{RUN_TAG}"
)

TABLES_DIR = os.path.join(BASE_OUTPUT_DIR, "tables")
PLOTS_DIR = os.path.join(BASE_OUTPUT_DIR, "plots")
MODELS_DIR = os.path.join(BASE_OUTPUT_DIR, "models")

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

all_results = []

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
print("FP16:", USE_FP16)
print("Checkpoint:", MODEL_CHECKPOINT)
print("BASE_OUTPUT_DIR:", BASE_OUTPUT_DIR)
print("TABLES_DIR:", TABLES_DIR)
print("PLOTS_DIR:", PLOTS_DIR)
print("MODELS_DIR:", MODELS_DIR)

print("\nRun mode:")
print({
    "FAST_RUN": FAST_RUN,
    "DEBUG_MODE": DEBUG_MODE,
    "RUN_NAME": RUN_NAME,
    "SCRATCH_EPOCHS": SCRATCH_EPOCHS,
})

print("\nEpochs:")
print({
    "FT_EPOCHS": FT_EPOCHS,
    "LORA_EPOCHS": LORA_EPOCHS,
    "JOINT_EPOCHS": JOINT_EPOCHS,
    "ORTH_EPOCHS": ORTH_EPOCHS,
    "RANKEXT_EPOCHS": RANKEXT_EPOCHS,
})

print("\nLoRA:")
print({
    "LORA_R": LORA_R,
    "LORA_ALPHA": LORA_ALPHA,
    "LORA_DROPOUT": LORA_DROPOUT,
    "TARGET_MODULES": TARGET_MODULES,
})

print("\nOrthogonal loss:")
print({
    "LAMBDA_ORTH_TRACE_LIST": LAMBDA_ORTH_TRACE_LIST,
    "LAMBDA_ORTH_NORM_LIST": LAMBDA_ORTH_NORM_LIST,
    "ORTH_NORM_EPS": ORTH_NORM_EPS,
    "ORTH_LOSS_LOG_EVERY": ORTH_LOSS_LOG_EVERY,
    "USE_IPC_CONSTRAINT": USE_IPC_CONSTRAINT,
    "ORTH_SCALE_MODE": ORTH_SCALE_MODE,
    "ORTH_TARGET_RATIO": ORTH_TARGET_RATIO,
})

print("\nRank extension:")
print({
    "RANKEXT_NEW_RANK_PER_STEP": RANKEXT_NEW_RANK_PER_STEP,
    "RANKEXT_ALPHA_PER_RANK": RANKEXT_ALPHA_PER_RANK,
    "LR_RANKEXT": LR_RANKEXT,
    "RANKEXT_REPLAY_PER_CLASS": RANKEXT_REPLAY_PER_CLASS,
    "RANKEXT_DIAGNOSTICS": RANKEXT_DIAGNOSTICS,
})

print("\nMethods:")
print(json.dumps(METHODS_TO_RUN, indent=2))

enabled_methods = [m for m, enabled in METHODS_TO_RUN.items() if enabled]
print("Enabled methods for this run:", enabled_methods)


In [ ]:
# ============================================================
# 3) Load CIFAR-100 and define continual splits
# ============================================================

dataset = load_dataset("cifar100")

LABEL_COL = "fine_label" if "fine_label" in dataset["train"].column_names else "label"
IMAGE_COL = "img" if "img" in dataset["train"].column_names else "image"

class_splits = [
    list(range(0, 20)),
    list(range(20, 40)),
    list(range(40, 60)),
    list(range(60, 80)),
    list(range(80, 100)),
]

first_step_classes = class_splits[0]
later_step_classes = [c for split in class_splits[1:] for c in split]
all_classes = [c for split in class_splits for c in split]

def classes_for_step(step_idx):
    return class_splits[step_idx]

def filter_by_classes(ds, class_ids):
    class_ids = set(class_ids)
    return ds.filter(lambda x: int(x[LABEL_COL]) in class_ids)

print("Dataset columns:", dataset["train"].column_names)
print("Label column:", LABEL_COL)
print("Image column:", IMAGE_COL)
for i, cls in enumerate(class_splits, start=1):
    print(f"Step {i}: {cls[0]}-{cls[-1]}")

In [ ]:
# ============================================================
# 4) CLIP image processor and transforms
# ============================================================

image_processor = CLIPImageProcessor.from_pretrained(MODEL_CHECKPOINT)

if hasattr(image_processor, "crop_size") and image_processor.crop_size is not None:
    H = int(image_processor.crop_size.get("height", 224))
    W = int(image_processor.crop_size.get("width", 224))
else:
    H = W = 224

train_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.RandomCrop((H, W), padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(
        brightness=0.05,
        contrast=0.05,
        saturation=0.05,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=image_processor.image_mean,
        std=image_processor.image_std,
    ),
])

val_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=image_processor.image_mean,
        std=image_processor.image_std,
    ),
])

def to_pil(x):
    if isinstance(x, Image.Image):
        return x.convert("RGB")

    if isinstance(x, dict):
        if "array" in x:
            x = x["array"]
        elif "bytes" in x:
            import io
            return Image.open(io.BytesIO(x["bytes"])).convert("RGB")

    if isinstance(x, list):
        x = np.array(x, dtype=np.uint8)

    if isinstance(x, np.ndarray):
        arr = np.squeeze(x).astype(np.uint8)

        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)

        if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[-1] not in (1, 3):
            arr = np.transpose(arr, (1, 2, 0))

        if arr.ndim == 3 and arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)

        return Image.fromarray(arr).convert("RGB")

    return x

def preprocess_train(ex):
    ex["pixel_values"] = [train_transform(to_pil(img)) for img in ex[IMAGE_COL]]
    ex["labels"] = [int(y) for y in ex[LABEL_COL]]
    return ex

def preprocess_val(ex):
    ex["pixel_values"] = [val_transform(to_pil(img)) for img in ex[IMAGE_COL]]
    ex["labels"] = [int(y) for y in ex[LABEL_COL]]
    return ex

def collate_fn(examples):
    pixel_values = torch.stack([e["pixel_values"] for e in examples])
    labels = torch.tensor([int(e["labels"]) for e in examples], dtype=torch.long)
    return {
        "pixel_values": pixel_values,
        "labels": labels,
    }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": float((preds == labels).mean())
    }

print("Image size:", H, W)
print("CLIP mean:", image_processor.image_mean)
print("CLIP std:", image_processor.image_std)

In [ ]:
# ============================================================
# 5) CLIP-ViT vision classifier wrapper
# ============================================================

class CLIPVisionForCIFAR100(nn.Module):
    """
    CLIP-ViT vision encoder + trainable CIFAR-100 classifier.

    This uses:
    openai/clip-vit-base-patch16

    The text encoder is not used.
    Only the CLIP vision backbone is used.
    """

    def __init__(self, checkpoint, num_labels):
        super().__init__()

        self.vision_model = CLIPVisionModel.from_pretrained(checkpoint, use_safetensors=True)

        hidden_size = self.vision_model.config.hidden_size
        self.classifier = nn.Linear(hidden_size, num_labels)

        self.config = self.vision_model.config
        self.config.num_labels = num_labels
        self.config.id2label = {i: str(i) for i in range(num_labels)}
        self.config.label2id = {str(i): i for i in range(num_labels)}

    def forward(
        self,
        pixel_values=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=True,
        **kwargs,
    ):
        outputs = self.vision_model(
            pixel_values=pixel_values,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=True,
        )

        pooled_output = outputs.pooler_output
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)

        return ImageClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

def fresh_pretrained_model():
    """
    Fresh CLIP-ViT vision model with a CIFAR-100 classifier.
    """
    return CLIPVisionForCIFAR100(
        checkpoint=MODEL_CHECKPOINT,
        num_labels=NUM_CLASSES,
    )

def disable_incompatible_torchao_for_peft():
    """
    Colab may have an old torchao installed. Recent PEFT checks torchao during
    LoRA injection and raises before falling back to normal nn.Linear LoRA.
    This guard disables only PEFT's torchao LoRA dispatcher when that version
    check fails; it does not change the LoRA method.
    """
    try:
        import peft.import_utils as peft_import_utils

        try:
            peft_import_utils.is_torchao_available()
            return
        except ImportError as e:
            if "incompatible version of torchao" not in str(e):
                raise

        peft_import_utils.is_torchao_available = lambda: False

        try:
            import peft.tuners.lora.torchao as peft_lora_torchao
            peft_lora_torchao.is_torchao_available = lambda: False
        except Exception:
            pass

        print(
            "[PEFT compatibility] Disabled torchao LoRA dispatcher "
            "because installed torchao is incompatible with PEFT."
        )
    except ImportError:
        return

def add_lora(model):
    """
    Add LoRA to CLIP-ViT attention q_proj and v_proj modules.
    """
    disable_incompatible_torchao_for_peft()

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        modules_to_save=["classifier"],
    )

    model = get_peft_model(model, lora_config)
    return model

print("LoRA target modules:", TARGET_MODULES)


In [ ]:
# ============================================================
# 6) Dataset helpers
# ============================================================

def build_replay_dataset(old_classes, replay_per_class):
    if len(old_classes) == 0 or replay_per_class <= 0:
        return None

    parts = []

    for cls in old_classes:
        cls_ds = filter_by_classes(dataset["train"], [cls])
        n = min(replay_per_class, len(cls_ds))
        cls_ds = cls_ds.shuffle(seed=SEED).select(range(n))
        parts.append(cls_ds)

    replay_ds = concatenate_datasets(parts)
    return replay_ds

def make_train_dataset(step_idx, replay_per_class=0):
    current_classes = classes_for_step(step_idx)
    current_ds = filter_by_classes(dataset["train"], current_classes)

    old_classes = []
    for old_step in range(step_idx):
        old_classes.extend(classes_for_step(old_step))

    replay_ds = build_replay_dataset(
        old_classes=old_classes,
        replay_per_class=replay_per_class,
    )

    if replay_ds is None:
        final_ds = current_ds
    else:
        final_ds = concatenate_datasets([current_ds, replay_ds])

    final_ds = final_ds.shuffle(seed=SEED + step_idx)
    final_ds = final_ds.with_transform(preprocess_train)

    print(
        f"Step {step_idx + 1} | "
        f"current={len(current_ds)} | "
        f"replay={0 if replay_ds is None else len(replay_ds)} | "
        f"total={len(final_ds)}"
    )

    return final_ds

def make_eval_dataset(class_ids):
    ds = filter_by_classes(dataset["test"], class_ids)
    ds = ds.with_transform(preprocess_val)
    return ds

def make_joint_train_dataset():
    ds = dataset["train"].shuffle(seed=SEED)
    ds = ds.with_transform(preprocess_train)
    return ds

def make_joint_eval_dataset():
    ds = dataset["test"]
    ds = ds.with_transform(preprocess_val)
    return ds

eval_first = make_eval_dataset(first_step_classes)
eval_later = make_eval_dataset(later_step_classes)
eval_all_seen = make_eval_dataset(all_classes)

print("first_step eval:", len(eval_first))
print("later_steps eval:", len(eval_later))
print("all_seen eval:", len(eval_all_seen))

In [ ]:
# ============================================================
# 7) Trainer helpers
# ============================================================

def get_training_args(
    output_dir,
    epochs,
    lr,
    batch_size,
    accum_steps,
    train_dataset_len=None,
    eval_strategy="epoch",
):
    """
    Trainer settings.

    We use warmup_steps instead of warmup_ratio because warmup_ratio is deprecated
    in newer Transformers versions.
    """

    if train_dataset_len is not None:
        steps_per_epoch = math.ceil(train_dataset_len / batch_size / accum_steps)
        total_steps = int(steps_per_epoch * epochs)
        warmup_steps = int(WARMUP_RATIO * total_steps)
    else:
        warmup_steps = 0

    kwargs = dict(
        output_dir=output_dir,
        remove_unused_columns=False,
        save_strategy="no",
        num_train_epochs=epochs,
        learning_rate=lr,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=warmup_steps,
        lr_scheduler_type=SCHED,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=accum_steps,
        fp16=USE_FP16,
        dataloader_num_workers=4,
        logging_steps=50,
        report_to="none",
        max_grad_norm=1.0,
    )

    sig = inspect.signature(TrainingArguments.__init__)

    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = eval_strategy
    else:
        kwargs["evaluation_strategy"] = eval_strategy

    return TrainingArguments(**kwargs)

def train_with_trainer(
    model,
    train_ds,
    eval_ds,
    output_dir,
    epochs,
    lr,
    batch_size,
    accum_steps,
    trainer_cls=Trainer,
    **trainer_kwargs,
):
    args = get_training_args(
        output_dir=output_dir,
        epochs=epochs,
        lr=lr,
        batch_size=batch_size,
        accum_steps=accum_steps,
        train_dataset_len=len(train_ds),
        eval_strategy="epoch",
    )

    trainer = trainer_cls(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
        **trainer_kwargs,
    )

    trainer.train()
    eval_out = trainer.evaluate()

    return trainer, eval_out

def evaluate_model(model, method_name):
    args = get_training_args(
        output_dir=os.path.join(MODELS_DIR, f"eval_{method_name}"),
        epochs=1,
        lr=LR_LORA,
        batch_size=BATCH_LORA,
        accum_steps=ACCUM_LORA,
        train_dataset_len=None,
        eval_strategy="no",
    )

    trainer = Trainer(
        model=model,
        args=args,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    eval_first_out = trainer.evaluate(eval_dataset=eval_first)
    eval_later_out = trainer.evaluate(eval_dataset=eval_later)
    eval_all_out = trainer.evaluate(eval_dataset=eval_all_seen)

    rows = [
        {
            "method": method_name,
            "eval_set": "first_step",
            "accuracy": float(eval_first_out["eval_accuracy"]),
            "loss": float(eval_first_out["eval_loss"]),
        },
        {
            "method": method_name,
            "eval_set": "later_steps",
            "accuracy": float(eval_later_out["eval_accuracy"]),
            "loss": float(eval_later_out["eval_loss"]),
        },
        {
            "method": method_name,
            "eval_set": "all_seen",
            "accuracy": float(eval_all_out["eval_accuracy"]),
            "loss": float(eval_all_out["eval_loss"]),
        },
    ]

    all_results.extend(rows)

    print(pd.DataFrame(rows))
    return rows

In [ ]:
# ============================================================
# 8) LoRA extraction and merge helpers
# ============================================================

def normalize_module_name(name):
    prefixes = [
        "base_model.model.",
        "model.",
    ]

    out = name

    for p in prefixes:
        if out.startswith(p):
            out = out[len(p):]

    return out

def extract_lora_state(model):
    """
    Extract:
    - LoRA delta_W per target module
    - classifier weights

    PEFT convention:
    A shape = [r, in_features]
    B shape = [out_features, r]
    delta_W = B @ A * scaling
    """
    state = {
        "deltas": {},
        "lora_A": {},
        "lora_B": {},
        "scaling": {},
        "classifier_weight": None,
        "classifier_bias": None,
    }

    for name, module in model.named_modules():
        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )

        if not has_lora:
            continue

        adapter_name = "default"
        A = module.lora_A[adapter_name].weight.detach().cpu().float().clone()
        B = module.lora_B[adapter_name].weight.detach().cpu().float().clone()

        scaling = (
            module.scaling[adapter_name]
            if isinstance(module.scaling, dict)
            else module.scaling
        )

        scaling = float(scaling)
        delta = scaling * (B @ A)

        plain_name = normalize_module_name(name)
        state["deltas"][plain_name] = delta.clone()
        state["lora_A"][plain_name] = A
        state["lora_B"][plain_name] = B
        state["scaling"][plain_name] = scaling

    for name, tensor in model.state_dict().items():
        if "classifier.modules_to_save.default.weight" in name:
            state["classifier_weight"] = tensor.detach().cpu().clone()

        if "classifier.modules_to_save.default.bias" in name:
            state["classifier_bias"] = tensor.detach().cpu().clone()

    return state

def get_submodule_by_name(model, module_name):
    module_name = normalize_module_name(module_name)

    current = model

    for part in module_name.split("."):
        if part == "":
            continue
        current = getattr(current, part)

    return current

def simple_average_deltas(step_states):
    keys = sorted(step_states[0]["deltas"].keys())
    merged = {}

    for key in keys:
        vals = []

        for state in step_states:
            if key in state["deltas"]:
                vals.append(state["deltas"][key].float())

        merged[key] = torch.stack(vals, dim=0).mean(dim=0)

    return merged

def column_normalize(mat, eps=1e-12):
    return mat / torch.linalg.norm(mat, dim=0, keepdim=True).clamp_min(eps)

def column_decouple_delta(delta, eps=1e-12):
    magnitudes = torch.linalg.norm(delta, dim=0, keepdim=True).clamp_min(eps)
    directions = delta / magnitudes
    return magnitudes, directions

def mean_pairwise_cosine(flat_vectors, eps=1e-12):
    if len(flat_vectors) < 2:
        return None

    sims = []

    for i in range(len(flat_vectors)):
        vi = flat_vectors[i]
        vi = vi / torch.linalg.norm(vi).clamp_min(eps)

        for j in range(i + 1, len(flat_vectors)):
            vj = flat_vectors[j]
            vj = vj / torch.linalg.norm(vj).clamp_min(eps)
            sims.append(torch.dot(vi, vj).item())

    if len(sims) == 0:
        return None

    return float(sum(sims) / len(sims))

def orthogonalize_task_directions(task_deltas, eps=1e-12):
    mags = []
    dirs = []
    flat_dirs = []

    for delta in task_deltas:
        mag, direction = column_decouple_delta(delta, eps=eps)
        mags.append(mag)
        dirs.append(direction)
        flat_dirs.append(direction.reshape(-1))

    ortho_flat = []

    for v in flat_dirs:
        u = v.clone()

        for q in ortho_flat:
            u = u - torch.dot(u, q) * q

        n = torch.linalg.norm(u)

        if n < eps:
            u = v / torch.linalg.norm(v).clamp_min(eps)
        else:
            u = u / n

        ortho_flat.append(u)

    ortho_dirs = [
        column_normalize(u.reshape_as(dirs[i]), eps=eps)
        for i, u in enumerate(ortho_flat)
    ]

    return mags, ortho_dirs

def do_merge_deltas(step_states, eps=1e-12, use_orthogonalize=True, verbose=True):
    """
    DO-Merging-inspired: layer-wise orthogonalized, column-wise decoupled LoRA delta merging.
    """
    all_keys = sorted(set().union(*[set(s["deltas"].keys()) for s in step_states]))
    merged = {}

    layer_delta_counts = []
    cos_before_values = []
    cos_after_values = []
    col_mag_means = []
    col_mag_stds = []

    for key in all_keys:
        task_deltas = []

        for state in step_states:
            if key in state["deltas"]:
                task_deltas.append(state["deltas"][key].detach().cpu().float())

        if len(task_deltas) == 0:
            continue

        layer_delta_counts.append(len(task_deltas))

        mags_before = []
        dirs_before = []

        for delta in task_deltas:
            mag, direction = column_decouple_delta(delta, eps=eps)
            mags_before.append(mag)
            dirs_before.append(direction)

        flat_before = [d.reshape(-1) for d in dirs_before]
        cos_before = mean_pairwise_cosine(flat_before, eps=eps)

        if cos_before is not None:
            cos_before_values.append(cos_before)

        if len(task_deltas) == 1:
            merged[key] = task_deltas[0].clone()
            continue

        if use_orthogonalize:
            mags, dirs = orthogonalize_task_directions(task_deltas, eps=eps)
        else:
            mags = mags_before
            dirs = dirs_before

        flat_after = [d.reshape(-1) for d in dirs]
        cos_after = mean_pairwise_cosine(flat_after, eps=eps)

        if cos_after is not None:
            cos_after_values.append(cos_after)

        mag_stack = torch.stack(mags, dim=0)
        col_mag_means.append(float(mag_stack.mean().item()))
        col_mag_stds.append(float(mag_stack.std(unbiased=False).item()))

        merged_mag = mag_stack.mean(dim=0)
        merged_dir = torch.stack(dirs, dim=0).mean(dim=0)
        merged_dir = column_normalize(merged_dir, eps=eps)

        merged_delta = merged_dir * merged_mag

        if merged_delta.shape != task_deltas[0].shape:
            raise ValueError(
                f"Shape mismatch for {key}: merged={tuple(merged_delta.shape)} vs ref={tuple(task_deltas[0].shape)}"
            )

        merged[key] = merged_delta

    if verbose:
        print(f"[DO-Merging] merged {len(merged)} layers")

        if len(layer_delta_counts) > 0:
            mean_tasks = sum(layer_delta_counts) / len(layer_delta_counts)
            print(f"[DO-Merging] avg task deltas per layer: {mean_tasks:.2f}")

        if len(cos_before_values) > 0:
            print(
                f"[DO-Merging] avg pairwise cosine before orthogonalization: {sum(cos_before_values) / len(cos_before_values):.6f}"
            )
        else:
            print("[DO-Merging] avg pairwise cosine before orthogonalization: n/a")

        if len(cos_after_values) > 0:
            print(
                f"[DO-Merging] avg pairwise cosine after orthogonalization: {sum(cos_after_values) / len(cos_after_values):.6f}"
            )
        else:
            print("[DO-Merging] avg pairwise cosine after orthogonalization: n/a")

        if len(col_mag_means) > 0:
            print(
                f"[DO-Merging] column magnitude mean/std across layers: {sum(col_mag_means) / len(col_mag_means):.6f} / {sum(col_mag_stds) / len(col_mag_stds):.6f}"
            )
        else:
            print("[DO-Merging] column magnitude mean/std across layers: n/a")

    return merged

def apply_deltas_to_base(merged_deltas, step_states):
    """
    Apply merged LoRA deltas to a fresh CLIP-ViT model and stitch classifier rows.
    """
    model = fresh_pretrained_model()

    with torch.no_grad():
        for key, delta in merged_deltas.items():
            try:
                module = get_submodule_by_name(model, key)
            except Exception as e:
                print("Could not find module:", key, "|", e)
                continue

            if not hasattr(module, "weight"):
                print("Module has no weight:", key)
                continue

            module.weight.add_(
                delta.to(
                    device=module.weight.device,
                    dtype=module.weight.dtype,
                )
            )

        for step_idx, state in enumerate(step_states):
            classes = classes_for_step(step_idx)

            if state["classifier_weight"] is None:
                print("Missing classifier for step", step_idx + 1)
                continue

            w = state["classifier_weight"].to(model.classifier.weight.device)
            b = state["classifier_bias"].to(model.classifier.bias.device)

            for c in classes:
                model.classifier.weight[c].copy_(w[c])
                model.classifier.bias[c].copy_(b[c])

    return model

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# ============================================================
# 9) Baseline: full_finetune
# ============================================================

if METHODS_TO_RUN.get("full_finetune", False):
    full_ft_model = fresh_pretrained_model()

    for step_idx in range(NUM_STEPS):
        train_ds = make_train_dataset(step_idx, replay_per_class=0)
        eval_ds = make_eval_dataset(classes_for_step(step_idx))

        out_dir = os.path.join(
            MODELS_DIR,
            f"full_finetune_step_{step_idx + 1}",
        )

        print(
            f"\n===== full_finetune | "
            f"step {step_idx + 1}/{NUM_STEPS} ====="
        )

        train_with_trainer(
            model=full_ft_model,
            train_ds=train_ds,
            eval_ds=eval_ds,
            output_dir=out_dir,
            epochs=FT_EPOCHS,
            lr=LR_FT,
            batch_size=BATCH_FT,
            accum_steps=ACCUM_FT,
        )

    evaluate_model(full_ft_model, "full_finetune")

    del full_ft_model
    cleanup()

else:
    print("Skipping full_finetune")


In [ ]:
# ============================================================
# 10) Train independent LoRAs
# ============================================================

def extract_reference_weights_for_independent_lora_orth(peft_model):
    reference_weights = {}
    for name, module in peft_model.named_modules():
        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )
        if not has_lora:
            continue
        plain_name = normalize_module_name(name)
        if hasattr(module, "base_layer") and hasattr(module.base_layer, "weight"):
            reference_weights[plain_name] = module.base_layer.weight.detach().cpu().float().clone()
        elif hasattr(module, "weight"):
            reference_weights[plain_name] = module.weight.detach().cpu().float().clone()
    return reference_weights

def compute_independent_lora_orth_components(model, reference_weights, eps=1e-8):
    raw_trace_terms = []
    cosine_terms = []
    device = next(model.parameters()).device

    for name, module in model.named_modules():
        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )
        if not has_lora:
            continue

        plain_name = normalize_module_name(name)
        if plain_name not in reference_weights:
            continue

        A = module.lora_A["default"].weight
        B = module.lora_B["default"].weight
        scaling = module.scaling["default"] if isinstance(module.scaling, dict) else module.scaling
        delta = (B @ A) * float(scaling)
        old_weight = reference_weights[plain_name].to(device=delta.device, dtype=delta.dtype)

        raw_trace = torch.sum(old_weight * delta)
        delta_norm = torch.linalg.norm(delta).clamp_min(eps)
        ref_norm = torch.linalg.norm(old_weight).clamp_min(eps)
        cosine = raw_trace / (ref_norm * delta_norm)

        raw_trace_terms.append(raw_trace)
        cosine_terms.append(cosine)

    if len(cosine_terms) == 0:
        zero = torch.tensor(0.0, device=device)
        return {
            "num_layers": 0,
            "orth_loss_raw": zero,
            "orth_loss_abs": zero,
            "orth_loss_squared": zero,
            "mean_cosine_alignment": zero,
            "raw_trace_mean_unnormalized": zero,
        }

    raw_tensor = torch.stack(raw_trace_terms)
    cosine_tensor = torch.stack(cosine_terms)
    return {
        "num_layers": int(cosine_tensor.numel()),
        "orth_loss_raw": cosine_tensor.mean(),
        "orth_loss_abs": cosine_tensor.abs().mean(),
        "orth_loss_squared": cosine_tensor.pow(2).mean(),
        "mean_cosine_alignment": cosine_tensor.mean(),
        "raw_trace_mean_unnormalized": raw_tensor.mean(),
    }

class IndependentLoraOrthTrainer(Trainer):
    def __init__(
        self,
        *args,
        reference_weights=None,
        lambda_orth=1.0,
        orth_scale_mode="auto_ratio",
        orth_target_ratio=0.08,
        orth_lambda_min=1e-6,
        orth_lambda_max=1e3,
        orth_eps=1e-8,
        log_every_steps=200,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.reference_weights = reference_weights or {}
        self.lambda_orth = float(lambda_orth)
        self.orth_scale_mode = str(orth_scale_mode)
        self.orth_target_ratio = float(orth_target_ratio)
        self.orth_lambda_min = float(orth_lambda_min)
        self.orth_lambda_max = float(orth_lambda_max)
        self.orth_eps = float(orth_eps)
        self.log_every_steps = max(1, int(log_every_steps))

        self._ce_losses = []
        self._orth_raw_losses = []
        self._orth_abs_losses = []
        self._orth_squared_losses = []
        self._orth_used_losses = []
        self._orth_contrib_losses = []
        self._orth_ratio_losses = []
        self._effective_lambdas = []
        self._raw_trace_unnormalized = []
        self._total_losses = []

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        ce_loss = outputs.loss

        orth_comps = compute_independent_lora_orth_components(
            model=model,
            reference_weights=self.reference_weights,
            eps=self.orth_eps,
        )

        if self.orth_scale_mode == "raw_trace":
            orth_loss_used = orth_comps["orth_loss_raw"]
        elif self.orth_scale_mode == "abs_trace":
            orth_loss_used = orth_comps["orth_loss_abs"]
        elif self.orth_scale_mode == "squared_trace":
            orth_loss_used = orth_comps["orth_loss_squared"]
        elif self.orth_scale_mode == "cosine":
            orth_loss_used = orth_comps["orth_loss_raw"]
        elif self.orth_scale_mode == "auto_ratio":
            orth_loss_used = orth_comps["orth_loss_abs"]
        elif self.orth_scale_mode == "normalized_fro":
            orth_loss_used = orth_comps["orth_loss_abs"]
        else:
            raise ValueError(
                f"Unknown orth_scale_mode={self.orth_scale_mode}. "
                f"Use raw_trace, abs_trace, squared_trace, cosine, auto_ratio, or normalized_fro."
            )

        effective_lambda = torch.tensor(self.lambda_orth, device=ce_loss.device, dtype=ce_loss.dtype)
        if self.orth_scale_mode == "auto_ratio" and self.orth_target_ratio > 0.0:
            denom = orth_loss_used.detach().abs().clamp_min(self.orth_eps)
            target = float(self.orth_target_ratio) * ce_loss.detach()
            effective_lambda = target / denom
            effective_lambda = effective_lambda.clamp(min=self.orth_lambda_min, max=self.orth_lambda_max)
            effective_lambda = effective_lambda * float(self.lambda_orth)

        orth_contrib = effective_lambda * orth_loss_used
        loss = ce_loss + orth_contrib

        self._ce_losses.append(float(ce_loss.detach().cpu().item()))
        self._orth_raw_losses.append(float(orth_comps["orth_loss_raw"].detach().cpu().item()))
        self._orth_abs_losses.append(float(orth_comps["orth_loss_abs"].detach().cpu().item()))
        self._orth_squared_losses.append(float(orth_comps["orth_loss_squared"].detach().cpu().item()))
        self._orth_used_losses.append(float(orth_loss_used.detach().cpu().item()))
        self._orth_contrib_losses.append(float(orth_contrib.detach().cpu().item()))
        orth_ratio = abs(float(orth_contrib.detach().cpu().item())) / max(float(ce_loss.detach().cpu().item()), self.orth_eps)
        self._orth_ratio_losses.append(float(orth_ratio))
        self._effective_lambdas.append(float(effective_lambda.detach().cpu().item()))
        self._raw_trace_unnormalized.append(float(orth_comps["raw_trace_mean_unnormalized"].detach().cpu().item()))
        self._total_losses.append(float(loss.detach().cpu().item()))

        if len(self._ce_losses) % self.log_every_steps == 0:
            print(
                f"[independent_lora orth train] step={len(self._ce_losses)} | "
                f"ce={self._ce_losses[-1]:.6f} | "
                f"orth_raw={self._orth_raw_losses[-1]:.6f} | "
                f"orth_used={self._orth_used_losses[-1]:.6f} | "
                f"orth_weighted={self._orth_contrib_losses[-1]:.6f} | "
                f"orth_ratio={self._orth_ratio_losses[-1]:.6f} | "
                f"effective_lambda={self._effective_lambdas[-1]:.6f}"
            )

        return (loss, outputs) if return_outputs else loss

    def consume_logged_losses(self):
        if len(self._ce_losses) == 0:
            return None

        ce_mean = float(np.mean(self._ce_losses))
        orth_contrib_mean = float(np.mean(self._orth_contrib_losses))
        ratio_from_means = abs(orth_contrib_mean) / max(ce_mean, self.orth_eps)
        stats = {
            "ce_loss_mean": ce_mean,
            "orth_loss_raw_mean": float(np.mean(self._orth_raw_losses)),
            "orth_loss_abs_mean": float(np.mean(self._orth_abs_losses)),
            "orth_loss_squared_mean": float(np.mean(self._orth_squared_losses)),
            "orth_loss_used_mean": float(np.mean(self._orth_used_losses)),
            "orth_contribution_mean": orth_contrib_mean,
            "orth_contribution_ratio_mean": float(np.mean(self._orth_ratio_losses)),
            "orth_contribution_ratio_from_means": float(ratio_from_means),
            "effective_lambda_mean": float(np.mean(self._effective_lambdas)),
            "raw_trace_unnormalized_mean": float(np.mean(self._raw_trace_unnormalized)),
            "total_loss_mean": float(np.mean(self._total_losses)),
            "num_logged_steps": int(len(self._ce_losses)),
            "lambda_orth": float(self.lambda_orth),
            "orth_scale_mode": self.orth_scale_mode,
            "orth_target_ratio": float(self.orth_target_ratio),
        }

        self._ce_losses = []
        self._orth_raw_losses = []
        self._orth_abs_losses = []
        self._orth_squared_losses = []
        self._orth_used_losses = []
        self._orth_contrib_losses = []
        self._orth_ratio_losses = []
        self._effective_lambdas = []
        self._raw_trace_unnormalized = []
        self._total_losses = []
        return stats

def train_independent_loras(method_prefix, replay_per_class=0, use_orth=False, orth_train_records=None):
    """
    Train one independent LoRA per step.

    Used for:
    - simple_avg_no_replay
    - simple_avg_replay
    - do_merging_simple
    - do_merging_simple_orth
    """
    step_states = []

    for step_idx in range(NUM_STEPS):
        model = fresh_pretrained_model()
        model = add_lora(model)

        model.print_trainable_parameters()

        train_ds = make_train_dataset(
            step_idx=step_idx,
            replay_per_class=replay_per_class,
        )

        eval_ds = make_eval_dataset(classes_for_step(step_idx))

        out_dir = os.path.join(
            MODELS_DIR,
            f"{method_prefix}_step_{step_idx + 1}",
        )

        print(
            f"\n===== {method_prefix} | "
            f"step {step_idx + 1}/{NUM_STEPS} | "
            f"replay_per_class={replay_per_class} ====="
        )

        trainer_cls = Trainer
        trainer_kwargs = {}
        if use_orth:
            reference_weights = extract_reference_weights_for_independent_lora_orth(model)
            trainer_cls = IndependentLoraOrthTrainer
            trainer_kwargs = {
                "reference_weights": reference_weights,
                "lambda_orth": float(LAMBDA_ORTH),
                "orth_scale_mode": str(ORTH_SCALE_MODE),
                "orth_target_ratio": float(ORTH_TARGET_RATIO),
                "orth_lambda_min": float(ORTH_LAMBDA_MIN),
                "orth_lambda_max": float(ORTH_LAMBDA_MAX),
                "orth_eps": float(ORTH_EPS),
                "log_every_steps": int(ORTH_LOSS_LOG_EVERY),
            }

        trainer, _ = train_with_trainer(
            model=model,
            train_ds=train_ds,
            eval_ds=eval_ds,
            output_dir=out_dir,
            epochs=LORA_EPOCHS,
            lr=LR_LORA,
            batch_size=BATCH_LORA,
            accum_steps=ACCUM_LORA,
            trainer_cls=trainer_cls,
            **trainer_kwargs,
        )

        if use_orth and isinstance(trainer, IndependentLoraOrthTrainer):
            orth_stats = trainer.consume_logged_losses()
            if orth_stats is not None and orth_train_records is not None:
                orth_train_records.append({
                    "method": method_prefix,
                    "step": int(step_idx + 1),
                    "orth_loss_type": str(ORTH_LOSS_TYPE),
                    "orth_scale_mode": str(orth_stats["orth_scale_mode"]),
                    "orth_target_ratio": float(orth_stats["orth_target_ratio"]),
                    "lambda_orth": float(orth_stats["lambda_orth"]),
                    "effective_lambda_mean": float(orth_stats["effective_lambda_mean"]),
                    "ce_loss_mean": float(orth_stats["ce_loss_mean"]),
                    "orth_loss_raw_mean": float(orth_stats["orth_loss_raw_mean"]),
                    "orth_loss_abs_mean": float(orth_stats["orth_loss_abs_mean"]),
                    "orth_loss_squared_mean": float(orth_stats["orth_loss_squared_mean"]),
                    "orth_loss_used_mean": float(orth_stats["orth_loss_used_mean"]),
                    "orth_contribution_mean": float(orth_stats["orth_contribution_mean"]),
                    "orth_contribution_ratio_mean": float(orth_stats["orth_contribution_ratio_mean"]),
                    "orth_contribution_ratio_from_means": float(orth_stats["orth_contribution_ratio_from_means"]),
                    "raw_trace_unnormalized_mean": float(orth_stats["raw_trace_unnormalized_mean"]),
                    "total_loss_mean": float(orth_stats["total_loss_mean"]),
                    "logged_steps": int(orth_stats["num_logged_steps"]),
                })

        state = extract_lora_state(model)
        step_states.append(state)

        del model
        cleanup()

    return step_states

In [ ]:
# ============================================================
# 11) simple_avg (no replay)
# ============================================================

step_states_no_replay = None
step_states_no_replay_orth = None
independent_orth_train_rows = []

if METHODS_TO_RUN.get("simple_avg", False):
    step_states_no_replay = train_independent_loras(
        method_prefix="simple_avg_source",
        replay_per_class=0,
    )

    simple_delta = simple_average_deltas(step_states_no_replay)
    simple_model = apply_deltas_to_base(
        merged_deltas=simple_delta,
        step_states=step_states_no_replay,
    )

    evaluate_model(simple_model, "simple_avg")

    del simple_model
    cleanup()

else:
    print("Skipping simple_avg")

if len(independent_orth_train_rows) > 0:
    independent_orth_train_df = pd.DataFrame(independent_orth_train_rows)
    independent_orth_train_path = os.path.join(TABLES_DIR, "independent_lora_orth_train_losses.csv")
    independent_orth_train_df.to_csv(independent_orth_train_path, index=False)
    print("[independent_lora orth] saved training-loss diagnostics:", independent_orth_train_path)


In [ ]:
# ============================================================
# 12) simple_avg_replay
# ============================================================

step_states_replay = None

if METHODS_TO_RUN["simple_avg_replay"]:
    step_states_replay = train_independent_loras(
        method_prefix="simple_avg_replay_source",
        replay_per_class=REPLAY_PER_CLASS,
    )

    replay_delta = simple_average_deltas(step_states_replay)
    replay_model = apply_deltas_to_base(
        merged_deltas=replay_delta,
        step_states=step_states_replay,
    )

    evaluate_model(replay_model, "simple_avg_replay")

    del replay_model
    cleanup()

else:
    print("Skipping simple_avg_replay")

In [ ]:
# ============================================================
# 13) do_merging_simple
# ============================================================

if METHODS_TO_RUN["do_merging_simple"]:
    assert step_states_no_replay is not None, "step_states_no_replay is required for do_merging_simple"

    do_delta = do_merge_deltas(step_states_no_replay)
    do_layer_count = len(do_delta)
    expected_do_layers = 24  # CLIP ViT-B/16: 12 blocks x (q_proj, v_proj)
    print(f"[DO-Merging] merged layer count: {do_layer_count}")
    if abs(do_layer_count - expected_do_layers) > 2:
        print(
            f"[WARNING] do_merging_simple merged {do_layer_count} layers; expected around {expected_do_layers}."
        )
    do_model = apply_deltas_to_base(
        merged_deltas=do_delta,
        step_states=step_states_no_replay,
    )

    evaluate_model(do_model, "do_merging_simple")

    del do_model
    cleanup()

else:
    print("Skipping do_merging_simple")

if METHODS_TO_RUN.get("do_merging_simple_orth", False):
    assert step_states_no_replay_orth is not None, "step_states_no_replay_orth is required for do_merging_simple_orth"

    do_orth_delta = do_merge_deltas(step_states_no_replay_orth)
    do_orth_model = apply_deltas_to_base(
        merged_deltas=do_orth_delta,
        step_states=step_states_no_replay_orth,
    )

    evaluate_model(do_orth_model, "do_merging_simple_orth")

    del do_orth_model
    cleanup()

else:
    print("Skipping do_merging_simple_orth")

In [ ]:
# ============================================================
# 14) orthogonal_loss
# ============================================================

# L = CE + lambda_orth * mean_i tr(M_(t-1)^i (Delta W_t^i)^T)
# Here M_(t-1)^i is the previous absorbed model weight for the same q_proj/v_proj layer.
# Delta W_t^i is the current LoRA update B @ A for that layer.

def extract_reference_weights_for_orth(peft_model):
    """
    Extract M_(t-1) for every current LoRA target module.
    These are the base q_proj/v_proj weights before training the current LoRA.
    """
    reference_weights = {}

    for name, module in peft_model.named_modules():
        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )

        if not has_lora:
            continue

        plain_name = normalize_module_name(name)

        if hasattr(module, "base_layer") and hasattr(module.base_layer, "weight"):
            reference_weights[plain_name] = module.base_layer.weight.detach().cpu().float().clone()
        elif hasattr(module, "weight"):
            reference_weights[plain_name] = module.weight.detach().cpu().float().clone()

    return reference_weights

def compute_orth_penalty(model, reference_weights, eps=1e-8):
    """
    Mean normalized trace between previous absorbed model weights and current LoRA deltas.

    Orthogonality objective per layer:
        tr(M_(t-1) DeltaW_t^T)

    Since both matrices have the same shape, this is equivalent to:
        sum(M_(t-1) * DeltaW_t)

    We normalize by matrix norms to keep the scale stable during debug runs.
    """
    penalties = []
    device = next(model.parameters()).device

    for name, module in model.named_modules():
        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )

        if not has_lora:
            continue

        plain_name = normalize_module_name(name)

        if plain_name not in reference_weights:
            continue

        A = module.lora_A["default"].weight
        B = module.lora_B["default"].weight

        scaling = (
            module.scaling["default"]
            if isinstance(module.scaling, dict)
            else module.scaling
        )

        delta = (B @ A) * float(scaling)
        old_weight = reference_weights[plain_name].to(
            device=delta.device,
            dtype=delta.dtype,
        )

        trace_value = torch.sum(old_weight * delta)
        normalized_trace = trace_value / (
            torch.linalg.norm(old_weight).clamp_min(eps)
            * torch.linalg.norm(delta).clamp_min(eps)
        )

        penalties.append(normalized_trace.pow(2))

    if not penalties:
        return torch.tensor(0.0, device=device)

    return torch.stack(penalties).mean()

def compute_orth_diagnostics(model, reference_weights, eps=1e-8):
    """
    Lightweight diagnostics for the current orthogonality objective.
    This does not change the training loss.
    """
    rows = []

    for name, module in model.named_modules():
        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )

        if not has_lora:
            continue

        plain_name = normalize_module_name(name)

        if plain_name not in reference_weights:
            continue

        A = module.lora_A["default"].weight
        B = module.lora_B["default"].weight

        scaling = (
            module.scaling["default"]
            if isinstance(module.scaling, dict)
            else module.scaling
        )

        delta = (B @ A) * float(scaling)
        old_weight = reference_weights[plain_name].to(
            device=delta.device,
            dtype=delta.dtype,
        )

        trace_value = torch.sum(old_weight * delta)
        normalized_trace = trace_value / (
            torch.linalg.norm(old_weight).clamp_min(eps)
            * torch.linalg.norm(delta).clamp_min(eps)
        )

        rows.append({
            "layer": plain_name,
            "raw_trace": float(trace_value.detach().cpu()),
            "normalized_trace": float(normalized_trace.detach().cpu()),
            "squared_penalty": float(normalized_trace.pow(2).detach().cpu()),
            "delta_norm": float(torch.linalg.norm(delta).detach().cpu()),
            "reference_norm": float(torch.linalg.norm(old_weight).detach().cpu()),
        })

    if len(rows) == 0:
        print("[orth diagnostics] no matched LoRA/reference layers")
        return pd.DataFrame()

    diag_df = pd.DataFrame(rows)
    summary = diag_df[
        [
            "raw_trace",
            "normalized_trace",
            "squared_penalty",
            "delta_norm",
            "reference_norm",
        ]
    ].mean()

    print("[orth diagnostics] mean over matched q_proj/v_proj layers")
    print(summary.round(6))

    return diag_df

class OrthogonalLossTrainer(Trainer):
    """
    Trainer for the orthogonal_loss variant.

    It adds an orthogonality-style penalty between:
    - previous absorbed model weights M_(t-1)
    - current LoRA delta DeltaW_t
    """

    def __init__(self, *args, reference_weights=None, lambda_orth=0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.reference_weights = reference_weights or {}
        self.lambda_orth = float(lambda_orth)

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        outputs = model(**inputs)
        ce_loss = outputs.loss

        orth_loss = compute_orth_penalty(
            model=model,
            reference_weights=self.reference_weights,
        )

        loss = ce_loss + self.lambda_orth * orth_loss

        return (loss, outputs) if return_outputs else loss

if METHODS_TO_RUN["orthogonal_loss"]:
    orth_model = fresh_pretrained_model()

    for step_idx in range(NUM_STEPS):
        print(f"\n===== orthogonal_loss | step {step_idx + 1}/{NUM_STEPS} =====")

        orth_peft_model = add_lora(orth_model)
        orth_peft_model.print_trainable_parameters()

        reference_weights = extract_reference_weights_for_orth(orth_peft_model)

        train_ds = make_train_dataset(step_idx, replay_per_class=0)
        eval_ds = make_eval_dataset(classes_for_step(step_idx))

        train_with_trainer(
            model=orth_peft_model,
            train_ds=train_ds,
            eval_ds=eval_ds,
            output_dir=os.path.join(MODELS_DIR, f"orthogonal_loss_step_{step_idx + 1}"),
            epochs=ORTH_EPOCHS,
            lr=LR_ORTH,
            batch_size=BATCH_LORA,
            accum_steps=ACCUM_LORA,
            trainer_cls=OrthogonalLossTrainer,
            reference_weights=reference_weights,
            lambda_orth=LAMBDA_ORTH,
        )

        if ORTH_DIAGNOSTICS:
            compute_orth_diagnostics(
                model=orth_peft_model,
                reference_weights=reference_weights,
            )

        orth_model = orth_peft_model.merge_and_unload()

        del orth_peft_model
        cleanup()

    evaluate_model(orth_model, "orthogonal_loss")

    del orth_model
    cleanup()

else:
    print("Skipping orthogonal_loss")


In [ ]:
# ============================================================
# 15) Rank extension: growing LoRA + zero-old-merge + delta orth variants
# ============================================================

from transformers import TrainerCallback


class GrowingRankLoRALinear(nn.Module):
    """
    One growing LoRA pair per layer.
    Frozen slice stores previous ranks; new slice is trainable.
    """

    def __init__(
        self,
        base_layer,
        total_rank,
        frozen_A=None,
        frozen_B=None,
        dropout=0.0,
        old_active_in_forward=True,
    ):
        super().__init__()
        self.base_layer = base_layer
        self.total_rank = int(total_rank)
        self.old_active_in_forward = bool(old_active_in_forward)

        if frozen_A is None or frozen_B is None:
            self.frozen_rank = 0
        else:
            if frozen_A.shape[0] != frozen_B.shape[1]:
                raise ValueError(
                    f"A/B frozen rank mismatch: A={tuple(frozen_A.shape)}, B={tuple(frozen_B.shape)}"
                )
            self.frozen_rank = int(frozen_A.shape[0])

        self.new_rank = self.total_rank - self.frozen_rank
        if self.new_rank < 0:
            raise ValueError(
                f"new_rank < 0 | total_rank={self.total_rank} frozen_rank={self.frozen_rank}"
            )

        self.in_features = base_layer.in_features
        self.out_features = base_layer.out_features

        for p in self.base_layer.parameters():
            p.requires_grad = False

        self.rankext_alpha = RANKEXT_ALPHA_PER_RANK * self.total_rank
        self.scaling = self.rankext_alpha / self.total_rank
        self.dropout = nn.Dropout(dropout)

        if self.frozen_rank > 0:
            self.A_frozen = nn.Parameter(frozen_A.detach().clone().float(), requires_grad=False)
            self.B_frozen = nn.Parameter(frozen_B.detach().clone().float(), requires_grad=False)
        else:
            self.A_frozen = None
            self.B_frozen = None

        if self.new_rank > 0:
            self.A_new = nn.Parameter(torch.zeros(self.new_rank, self.in_features))
            self.B_new = nn.Parameter(torch.zeros(self.out_features, self.new_rank))
            nn.init.kaiming_uniform_(self.A_new, a=np.sqrt(5))
            nn.init.zeros_(self.B_new)
        else:
            self.A_new = None
            self.B_new = None

    def full_A_B(self):
        A_parts = []
        B_parts = []
        if self.frozen_rank > 0:
            A_parts.append(self.A_frozen.to(device=self.base_layer.weight.device, dtype=self.base_layer.weight.dtype))
            B_parts.append(self.B_frozen.to(device=self.base_layer.weight.device, dtype=self.base_layer.weight.dtype))
        if self.new_rank > 0:
            A_parts.append(self.A_new)
            B_parts.append(self.B_new)
        if len(A_parts) == 0:
            raise ValueError("No LoRA blocks available in full_A_B.")
        A = torch.cat(A_parts, dim=0)
        B = torch.cat(B_parts, dim=1)
        return A, B

    def current_new_delta(self):
        if self.new_rank <= 0:
            return None
        return (self.B_new @ self.A_new) * float(self.scaling)

    def cumulative_old_delta(self):
        if self.frozen_rank <= 0:
            return None
        return (self.B_frozen @ self.A_frozen) * float(self.scaling)

    def forward(self, x):
        base_out = self.base_layer(x)
        x_dropped = self.dropout(x)
        out = base_out

        if self.old_active_in_forward and self.frozen_rank > 0:
            hidden_old = torch.matmul(x_dropped, self.A_frozen.T)
            lora_old = torch.matmul(hidden_old, self.B_frozen.T)
            out = out + self.scaling * lora_old

        if self.new_rank > 0:
            hidden_new = torch.matmul(x_dropped, self.A_new.T)
            lora_new = torch.matmul(hidden_new, self.B_new.T)
            out = out + self.scaling * lora_new

        return out


def get_parent_module_and_child_name(model, module_name):
    parts = module_name.split(".")
    parent = model
    for p in parts[:-1]:
        parent = getattr(parent, p)
    return parent, parts[-1]


def find_clip_target_linear_modules(model):
    target_names = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and (name.endswith("q_proj") or name.endswith("v_proj")):
            target_names.append(name)
    return target_names


def build_rank_extension_model(previous_rank_state=None, step_idx=0, old_active_in_forward=True):
    model = fresh_pretrained_model()

    for _, p in model.vision_model.named_parameters():
        p.requires_grad = False
    for p in model.classifier.parameters():
        p.requires_grad = True

    total_rank = RANKEXT_NEW_RANK_PER_STEP * (step_idx + 1)
    target_names = find_clip_target_linear_modules(model)
    model._rank_extension_target_names = list(target_names)
    model._rank_extension_old_active_in_forward = bool(old_active_in_forward)

    print(f"[rank_extension] Step {step_idx + 1}")
    print(f"  total_rank: {total_rank}")
    print(f"  target linear modules: {len(target_names)}")
    print(f"  old_active_in_forward: {bool(old_active_in_forward)}")

    for module_name in target_names:
        parent, child_name = get_parent_module_and_child_name(model, module_name)
        base_layer = getattr(parent, child_name)

        frozen_A = None
        frozen_B = None
        if previous_rank_state is not None and module_name in previous_rank_state["lora"]:
            frozen_A = previous_rank_state["lora"][module_name]["A"]
            frozen_B = previous_rank_state["lora"][module_name]["B"]

        setattr(
            parent,
            child_name,
            GrowingRankLoRALinear(
                base_layer=base_layer,
                total_rank=total_rank,
                frozen_A=frozen_A,
                frozen_B=frozen_B,
                dropout=LORA_DROPOUT,
                old_active_in_forward=old_active_in_forward,
            ),
        )

    if previous_rank_state is not None and previous_rank_state["classifier_weight"] is not None:
        with torch.no_grad():
            model.classifier.weight.copy_(
                previous_rank_state["classifier_weight"].to(
                    device=model.classifier.weight.device,
                    dtype=model.classifier.weight.dtype,
                )
            )
            model.classifier.bias.copy_(
                previous_rank_state["classifier_bias"].to(
                    device=model.classifier.bias.device,
                    dtype=model.classifier.bias.dtype,
                )
            )

    return model


def extract_rank_extension_state(model):
    state = {"lora": {}, "classifier_weight": None, "classifier_bias": None}
    for name, module in model.named_modules():
        if isinstance(module, GrowingRankLoRALinear):
            A, B = module.full_A_B()
            state["lora"][name] = {
                "A": A.detach().cpu().clone(),
                "B": B.detach().cpu().clone(),
                "scaling": float(module.scaling),
                "total_rank": int(module.total_rank),
                "frozen_rank": int(module.frozen_rank),
                "new_rank": int(module.new_rank),
                "rankext_alpha": float(module.rankext_alpha),
            }
    state["classifier_weight"] = model.classifier.weight.detach().cpu().clone()
    state["classifier_bias"] = model.classifier.bias.detach().cpu().clone()
    return state


def rank_extension_trainable_classifier_classes(step_idx, replay_per_class):
    classes = list(classes_for_step(step_idx))
    if replay_per_class > 0:
        for old_step in range(step_idx):
            classes.extend(classes_for_step(old_step))
    return sorted(set(int(c) for c in classes))


def add_classifier_row_gradient_mask(model, trainable_classes):
    trainable_classes = set(int(c) for c in trainable_classes)
    mask_w = torch.zeros_like(model.classifier.weight)
    mask_b = torch.zeros_like(model.classifier.bias)
    for c in trainable_classes:
        mask_w[c, :] = 1.0
        mask_b[c] = 1.0
    hook_w = model.classifier.weight.register_hook(
        lambda grad: grad * mask_w.to(device=grad.device, dtype=grad.dtype)
    )
    hook_b = model.classifier.bias.register_hook(
        lambda grad: grad * mask_b.to(device=grad.device, dtype=grad.dtype)
    )
    return [hook_w, hook_b]


def snapshot_protected_classifier_rows(model, trainable_classes):
    trainable_classes = set(int(c) for c in trainable_classes)
    protected_rows = [c for c in range(NUM_CLASSES) if c not in trainable_classes]
    return {
        "rows": protected_rows,
        "weight": model.classifier.weight.detach().cpu().clone(),
        "bias": model.classifier.bias.detach().cpu().clone(),
    }


def restore_protected_classifier_rows(model, snapshot):
    rows = snapshot["rows"]
    if len(rows) == 0:
        return
    with torch.no_grad():
        row_idx = torch.tensor(rows, device=model.classifier.weight.device, dtype=torch.long)
        model.classifier.weight[row_idx].copy_(
            snapshot["weight"][rows].to(
                device=model.classifier.weight.device,
                dtype=model.classifier.weight.dtype,
            )
        )
        model.classifier.bias[row_idx].copy_(
            snapshot["bias"][rows].to(
                device=model.classifier.bias.device,
                dtype=model.classifier.bias.dtype,
            )
        )


def classifier_protected_row_max_diff(model, snapshot):
    rows = snapshot["rows"]
    if len(rows) == 0:
        return 0.0
    with torch.no_grad():
        weight_diff = (
            model.classifier.weight.detach().cpu()[rows] - snapshot["weight"][rows]
        ).abs().max().item()
        bias_diff = (
            model.classifier.bias.detach().cpu()[rows] - snapshot["bias"][rows]
        ).abs().max().item()
    return max(weight_diff, bias_diff)


class ClassifierRowRestoreCallback(TrainerCallback):
    def __init__(self, snapshot):
        self.snapshot = snapshot

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if model is not None:
            restore_protected_classifier_rows(model, self.snapshot)
        return control

    def on_train_end(self, args, state, control, model=None, **kwargs):
        if model is not None:
            restore_protected_classifier_rows(model, self.snapshot)
        return control


class RankExtensionTrainer(Trainer):
    def __init__(self, *args, classifier_snapshot=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.classifier_snapshot = classifier_snapshot
        if classifier_snapshot is not None:
            self.add_callback(ClassifierRowRestoreCallback(classifier_snapshot))


def assert_rank_extension_structure(model, step_idx):
    expected_total_rank = RANKEXT_NEW_RANK_PER_STEP * (step_idx + 1)
    expected_frozen_rank = RANKEXT_NEW_RANK_PER_STEP * step_idx
    expected_new_rank = RANKEXT_NEW_RANK_PER_STEP
    expected_target_names = getattr(model, "_rank_extension_target_names", None)
    if expected_target_names is None:
        raise AssertionError("Missing _rank_extension_target_names on rank-extension model.")

    module_map = dict(model.named_modules())
    wrapped_names = [name for name, module in module_map.items() if isinstance(module, GrowingRankLoRALinear)]
    if sorted(wrapped_names) != sorted(expected_target_names):
        missing = sorted(set(expected_target_names) - set(wrapped_names))
        extra = sorted(set(wrapped_names) - set(expected_target_names))
        raise AssertionError(f"Rank-extension wrapper mismatch. missing={missing}, extra={extra}")

    for name in expected_target_names:
        module = module_map[name]
        if module.total_rank != expected_total_rank:
            raise AssertionError(f"{name} total_rank={module.total_rank}, expected={expected_total_rank}")
        if module.frozen_rank != expected_frozen_rank:
            raise AssertionError(f"{name} frozen_rank={module.frozen_rank}, expected={expected_frozen_rank}")
        if module.new_rank != expected_new_rank:
            raise AssertionError(f"{name} new_rank={module.new_rank}, expected={expected_new_rank}")
        if module.A_frozen is not None and module.A_frozen.requires_grad:
            raise AssertionError(f"{name}.A_frozen unexpectedly requires grad.")
        if module.B_frozen is not None and module.B_frozen.requires_grad:
            raise AssertionError(f"{name}.B_frozen unexpectedly requires grad.")
        if module.A_new is None or not module.A_new.requires_grad:
            raise AssertionError(f"{name}.A_new missing/frozen.")
        if module.B_new is None or not module.B_new.requires_grad:
            raise AssertionError(f"{name}.B_new missing/frozen.")

    trainable_lora_names = [
        name
        for name, p in model.named_parameters()
        if p.requires_grad and (".A_" in name or ".B_" in name)
    ]
    bad_lora_names = [
        name for name in trainable_lora_names
        if not (name.endswith(".A_new") or name.endswith(".B_new"))
    ]
    if bad_lora_names:
        raise AssertionError(f"Only A_new/B_new may be trainable LoRA params, got {bad_lora_names}")

    print(
        f"[rank_extension assertions] step={step_idx + 1} | "
        f"total_rank={expected_total_rank} | frozen_rank={expected_frozen_rank} | new_rank={expected_new_rank}"
    )


def snapshot_frozen_rank_blocks(model):
    snapshot = {}
    for name, module in model.named_modules():
        if isinstance(module, GrowingRankLoRALinear) and module.frozen_rank > 0:
            snapshot[name] = {
                "A": module.A_frozen.detach().cpu().clone(),
                "B": module.B_frozen.detach().cpu().clone(),
            }
    return snapshot


def check_frozen_rank_blocks_unchanged(model, snapshot, label, csv_path=None):
    if len(snapshot) == 0:
        print(f"[rank_extension diagnostics] {label}: no frozen rank blocks to compare")
        return pd.DataFrame()

    rows = []
    module_map = dict(model.named_modules())
    for name, before in snapshot.items():
        module = module_map[name]
        rows.append({
            "layer": name,
            "A_max_abs_diff": float((module.A_frozen.detach().cpu() - before["A"]).abs().max().item()),
            "B_max_abs_diff": float((module.B_frozen.detach().cpu() - before["B"]).abs().max().item()),
        })

    diag_df = pd.DataFrame(rows)
    max_a = float(diag_df["A_max_abs_diff"].max())
    max_b = float(diag_df["B_max_abs_diff"].max())
    print(f"[rank_extension diagnostics] {label}: max frozen A diff={max_a:.10f}, max frozen B diff={max_b:.10f}")

    if csv_path is not None:
        diag_df.to_csv(csv_path, index=False)
        print("[rank_extension diagnostics] saved frozen-block diagnostics:", csv_path)

    return diag_df


def compute_delta_orth_components(model, eps=1e-12):
    trace_terms = []
    abs_trace_terms = []
    norm_terms = []
    old_norm_terms = []
    new_norm_terms = []
    factor_a_terms = []
    factor_b_terms = []
    factor_total_terms = []
    a_overlap_terms = []
    b_overlap_terms = []
    rows = []
    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype

    for name, module in model.named_modules():
        if not isinstance(module, GrowingRankLoRALinear):
            continue
        if module.new_rank <= 0:
            continue

        old_delta = module.cumulative_old_delta()
        new_delta = module.current_new_delta()
        if new_delta is None:
            continue
        if old_delta is None:
            old_delta = torch.zeros_like(new_delta)

        inner = torch.sum(old_delta * new_delta)
        abs_inner = torch.abs(inner)
        old_sq = old_delta.norm(p="fro").pow(2)
        new_sq = new_delta.norm(p="fro").pow(2)
        denom = old_sq * new_sq + float(eps)
        norm_sq = inner.pow(2) / denom

        trace_terms.append(inner)
        abs_trace_terms.append(abs_inner)
        norm_terms.append(norm_sq)
        old_norm_terms.append(old_delta.norm(p="fro"))
        new_norm_terms.append(new_delta.norm(p="fro"))

        if module.frozen_rank > 0 and module.A_frozen is not None and module.B_frozen is not None:
            A_old = module.A_frozen.to(device=new_delta.device, dtype=new_delta.dtype)
            A_new = module.A_new
            B_old = module.B_frozen.to(device=new_delta.device, dtype=new_delta.dtype)
            B_new = module.B_new

            A_old_hat = A_old / A_old.norm(dim=1, keepdim=True).clamp_min(eps)
            A_new_hat = A_new / A_new.norm(dim=1, keepdim=True).clamp_min(eps)
            A_overlap = A_old_hat @ A_new_hat.T
            factor_a = torch.sum(A_overlap.pow(2))

            B_old_hat = B_old / B_old.norm(dim=0, keepdim=True).clamp_min(eps)
            B_new_hat = B_new / B_new.norm(dim=0, keepdim=True).clamp_min(eps)
            B_overlap = B_old_hat.T @ B_new_hat
            factor_b = torch.sum(B_overlap.pow(2))

            factor_total = factor_a + factor_b
            factor_a_terms.append(factor_a)
            factor_b_terms.append(factor_b)
            factor_total_terms.append(factor_total)
            a_overlap_terms.append(A_overlap.abs().mean())
            b_overlap_terms.append(B_overlap.abs().mean())
        rows.append({
            "layer": name,
            "inner_trace": float(inner.detach().cpu().item()),
            "abs_inner_trace": float(abs_inner.detach().cpu().item()),
            "old_norm": float(old_norm_terms[-1].detach().cpu().item()),
            "new_norm": float(new_norm_terms[-1].detach().cpu().item()),
            "norm_sq": float(norm_sq.detach().cpu().item()),
        })

    if len(trace_terms) == 0:
        zero = torch.tensor(0.0, device=device, dtype=dtype)
        return {
            "num_layers": 0,
            "trace_mean": zero,
            "trace_abs_mean": zero,
            "norm_sq_mean": zero,
            "old_norm_mean": zero,
            "new_norm_mean": zero,
            "factor_A_mean": zero,
            "factor_B_mean": zero,
            "factor_total_mean": zero,
            "mean_A_overlap": zero,
            "mean_B_overlap": zero,
            "diag_df": pd.DataFrame(),
        }

    return {
        "num_layers": int(len(trace_terms)),
        "trace_mean": torch.stack(trace_terms).mean(),
        "trace_abs_mean": torch.stack(abs_trace_terms).mean(),
        "norm_sq_mean": torch.stack(norm_terms).mean(),
        "old_norm_mean": torch.stack(old_norm_terms).mean(),
        "new_norm_mean": torch.stack(new_norm_terms).mean(),
        "factor_A_mean": (torch.stack(factor_a_terms).mean() if len(factor_a_terms) > 0 else torch.tensor(0.0, device=device, dtype=dtype)),
        "factor_B_mean": (torch.stack(factor_b_terms).mean() if len(factor_b_terms) > 0 else torch.tensor(0.0, device=device, dtype=dtype)),
        "factor_total_mean": (torch.stack(factor_total_terms).mean() if len(factor_total_terms) > 0 else torch.tensor(0.0, device=device, dtype=dtype)),
        "mean_A_overlap": (torch.stack(a_overlap_terms).mean() if len(a_overlap_terms) > 0 else torch.tensor(0.0, device=device, dtype=dtype)),
        "mean_B_overlap": (torch.stack(b_overlap_terms).mean() if len(b_overlap_terms) > 0 else torch.tensor(0.0, device=device, dtype=dtype)),
        "diag_df": pd.DataFrame(rows),
    }


def cumulative_orth_formula_label(step_idx):
    t = int(step_idx + 1)
    if t <= 1:
        return "orth disabled at step 1 (no previous delta)"
    old_terms = " + ".join([f"L{i}" for i in range(1, t)])
    return f"orth({old_terms}, L{t})"


def collect_rank_block_grad_norms(model, train_ds):
    if len(train_ds) == 0:
        return {
            "frozen_A_grad_norm_mean": 0.0,
            "frozen_B_grad_norm_mean": 0.0,
            "new_A_grad_norm_mean": 0.0,
            "new_B_grad_norm_mean": 0.0,
        }

    device = next(model.parameters()).device
    model.train()
    model.zero_grad(set_to_none=True)
    loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=1,
        shuffle=False,
        collate_fn=collate_fn,
    )
    batch = next(iter(loader))
    batch = {
        k: (v.to(device) if torch.is_tensor(v) else v)
        for k, v in batch.items()
    }

    out = model(**batch)
    out.loss.backward()

    frozen_a = []
    frozen_b = []
    new_a = []
    new_b = []

    for _, module in model.named_modules():
        if not isinstance(module, GrowingRankLoRALinear):
            continue
        if module.A_frozen is not None:
            g = module.A_frozen.grad
            frozen_a.append(0.0 if g is None else float(g.norm().detach().cpu().item()))
        if module.B_frozen is not None:
            g = module.B_frozen.grad
            frozen_b.append(0.0 if g is None else float(g.norm().detach().cpu().item()))
        if module.A_new is not None:
            g = module.A_new.grad
            new_a.append(0.0 if g is None else float(g.norm().detach().cpu().item()))
        if module.B_new is not None:
            g = module.B_new.grad
            new_b.append(0.0 if g is None else float(g.norm().detach().cpu().item()))

    model.zero_grad(set_to_none=True)

    def _mean(xs):
        return float(np.mean(xs)) if len(xs) > 0 else 0.0

    return {
        "frozen_A_grad_norm_mean": _mean(frozen_a),
        "frozen_B_grad_norm_mean": _mean(frozen_b),
        "new_A_grad_norm_mean": _mean(new_a),
        "new_B_grad_norm_mean": _mean(new_b),
    }


class DeltaOrthRankExtensionTrainer(RankExtensionTrainer):
    def __init__(
        self,
        *args,
        lambda_orth=0.0,
        orth_mode="trace_abs",
        orth_eps=1e-12,
        log_every_steps=1,
        method_name="unknown",
        step_idx=-1,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.lambda_orth = float(lambda_orth)
        self.orth_mode = str(orth_mode)
        self.orth_eps = float(orth_eps)
        self.log_every_steps = max(1, int(log_every_steps))
        self.method_name = str(method_name)
        self.step_idx = int(step_idx)
        self._rows = []

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        ce_loss = outputs.loss

        comps = compute_delta_orth_components(model=model, eps=self.orth_eps)
        raw_inner = comps["trace_mean"]
        abs_inner = comps["trace_abs_mean"]
        norm_sq = comps["norm_sq_mean"]

        if self.orth_mode in ["trace", "trace_abs"]:
            orth_loss_used = abs_inner
        elif self.orth_mode == "norm":
            orth_loss_used = norm_sq
        elif self.orth_mode == "factor_orth":
            orth_loss_used = comps["factor_total_mean"]
        else:
            raise ValueError(f"Unknown orth_mode={self.orth_mode}")

        weighted = float(self.lambda_orth) * orth_loss_used
        loss = ce_loss + weighted

        ce_v = float(ce_loss.detach().cpu().item())
        raw_inner_v = float(raw_inner.detach().cpu().item())
        abs_inner_v = float(abs_inner.detach().cpu().item())
        norm_sq_v = float(norm_sq.detach().cpu().item())
        orth_used_v = float(orth_loss_used.detach().cpu().item())
        weighted_v = float(weighted.detach().cpu().item())
        ratio_v = abs(weighted_v) / (ce_v + float(self.orth_eps))
        factor_a_v = float(comps["factor_A_mean"].detach().cpu().item())
        factor_b_v = float(comps["factor_B_mean"].detach().cpu().item())
        factor_total_v = float(comps["factor_total_mean"].detach().cpu().item())
        mean_a_overlap_v = float(comps["mean_A_overlap"].detach().cpu().item())
        mean_b_overlap_v = float(comps["mean_B_overlap"].detach().cpu().item())
        weighted_factor_v = float((float(self.lambda_orth) * comps["factor_total_mean"]).detach().cpu().item())
        weighted_factor_ratio_v = abs(weighted_factor_v) / (ce_v + float(self.orth_eps))
        epoch_val = float(self.state.epoch) if self.state.epoch is not None else np.nan
        row = {
            "method": self.method_name,
            "step": int(self.step_idx + 1),
            "epoch": epoch_val,
            "ce_loss": ce_v,
            "raw_inner": raw_inner_v,
            "abs_inner": abs_inner_v,
            "norm_sq_orth": norm_sq_v,
            "orth_loss_raw": raw_inner_v,
            "orth_loss_used": orth_used_v,
            "lambda_orth": float(self.lambda_orth),
            "lambda_orth_times_loss": weighted_v,
            "orth_ratio_abs_weighted_over_ce": ratio_v,
            "orth_mode": self.orth_mode,
            "num_layers_used": int(comps["num_layers"]),
            "old_norm_mean": float(comps["old_norm_mean"].detach().cpu().item()),
            "new_norm_mean": float(comps["new_norm_mean"].detach().cpu().item()),
            "factor_A_penalty_mean": factor_a_v,
            "factor_B_penalty_mean": factor_b_v,
            "factor_total_penalty_mean": factor_total_v,
            "weighted_factor_orth_mean": weighted_factor_v,
            "weighted_factor_orth_over_CE": weighted_factor_ratio_v,
            "mean_A_overlap": mean_a_overlap_v,
            "mean_B_overlap": mean_b_overlap_v,
            "effective_lambda": float(self.lambda_orth),
        }
        self._rows.append(row)

        if len(self._rows) % self.log_every_steps == 0:
            print(
                f"[orth train] method={self.method_name} | step={row['step']} | epoch={row['epoch']:.4f} | "
                f"ce={row['ce_loss']:.6f} | raw_inner={row['raw_inner']:.6f} | "
                f"orth_used={row['orth_loss_used']:.6f} | "
                f"lambda={row['lambda_orth']:.6g} | weighted={row['lambda_orth_times_loss']:.6f} | "
                f"ratio={row['orth_ratio_abs_weighted_over_ce']:.6f}"
            )

        return (loss, outputs) if return_outputs else loss

    def consume_logged_losses(self):
        if len(self._rows) == 0:
            return None
        out = pd.DataFrame(self._rows).copy()
        self._rows = []
        return out


def evaluate_seen_step_accuracies(model, upto_step_idx):
    args = get_training_args(
        output_dir=os.path.join(MODELS_DIR, "tmp_seen_eval"),
        epochs=1,
        lr=LR_LORA,
        batch_size=BATCH_LORA,
        accum_steps=ACCUM_LORA,
        train_dataset_len=None,
        eval_strategy="no",
    )
    trainer = Trainer(
        model=model,
        args=args,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )
    step_acc = {}
    for task_step in range(upto_step_idx + 1):
        eval_ds = make_eval_dataset(classes_for_step(task_step))
        out = trainer.evaluate(eval_dataset=eval_ds)
        step_acc[int(task_step)] = float(out["eval_accuracy"])
    return step_acc


def compute_average_forgetting(stepwise_task_accuracies):
    if len(stepwise_task_accuracies) == 0:
        return np.nan
    all_steps = sorted(stepwise_task_accuracies.keys())
    final_step = all_steps[-1]
    forgetting_values = []
    for task_step in all_steps:
        if task_step == final_step:
            continue
        vals = []
        for s in all_steps:
            if s < task_step:
                continue
            if task_step in stepwise_task_accuracies[s]:
                vals.append(stepwise_task_accuracies[s][task_step])
        if len(vals) == 0:
            continue
        best_acc = max(vals)
        final_acc = stepwise_task_accuracies[final_step].get(task_step, np.nan)
        if np.isnan(final_acc):
            continue
        forgetting_values.append(float(best_acc - final_acc))
    if len(forgetting_values) == 0:
        return np.nan
    return float(np.mean(forgetting_values))


def safe_lambda_tag(val):
    v = float(val)
    if abs(v - 1e-4) < 1e-12:
        return "lam_1em4"
    if abs(v - 1e-3) < 1e-12:
        return "lam_1em3"
    if abs(v - 0.01) < 1e-12:
        return "lam_001"
    if abs(v - 0.05) < 1e-12:
        return "lam_005"
    if abs(v - 0.1) < 1e-12:
        return "lam_01"
    s = f"{v:g}".replace("-", "m").replace(".", "p")
    return f"lam_{s}"


def print_rank_extension_step_diagnostics(
    method_name,
    step_idx,
    model,
    replay_per_class,
    old_active_in_forward,
    frozen_diff_df,
    grad_stats,
    eps=1e-12,
):
    modules = [m for _, m in model.named_modules() if isinstance(m, GrowingRankLoRALinear)]
    if len(modules) == 0:
        return
    ref = modules[0]
    total_rank = int(ref.total_rank)
    frozen_rank = int(ref.frozen_rank)
    new_rank = int(ref.new_rank)
    active_new_slice = (
        f"{frozen_rank + 1}-{frozen_rank + new_rank}" if new_rank > 0 else "none"
    )
    frozen_slice = f"1-{frozen_rank}" if frozen_rank > 0 else "none"

    comps = compute_delta_orth_components(model=model, eps=eps)
    raw_trace = float(comps["trace_mean"].detach().cpu().item())
    norm_sq = float(comps["norm_sq_mean"].detach().cpu().item())
    old_norm = float(comps["old_norm_mean"].detach().cpu().item())
    new_norm = float(comps["new_norm_mean"].detach().cpu().item())

    max_a_diff = 0.0
    max_b_diff = 0.0
    if len(frozen_diff_df) > 0:
        max_a_diff = float(frozen_diff_df["A_max_abs_diff"].max())
        max_b_diff = float(frozen_diff_df["B_max_abs_diff"].max())

    print(f"[rank_extension diagnostics] method={method_name} step={step_idx + 1}")
    print(f"  total_lora_rank={total_rank}")
    print(f"  active_trainable_rank_slice={active_new_slice}")
    print(f"  frozen_copied_rank_slice={frozen_slice}")
    print(f"  replay_disabled={(int(replay_per_class) == 0)}")
    print(f"  old_slices_active_in_forward={bool(old_active_in_forward)}")
    print(f"  cumulative_old_delta_norm={old_norm:.8f}")
    print(f"  current_new_delta_norm={new_norm:.8f}")
    print(f"  raw_trace_inner_sum(old*new)={raw_trace:.8f}")
    print(f"  normalized_squared_orth={norm_sq:.8f}")
    print(f"  frozen_A_grad_norm_mean={grad_stats['frozen_A_grad_norm_mean']:.8f}")
    print(f"  frozen_B_grad_norm_mean={grad_stats['frozen_B_grad_norm_mean']:.8f}")
    print(f"  new_A_grad_norm_mean={grad_stats['new_A_grad_norm_mean']:.8f}")
    print(f"  new_B_grad_norm_mean={grad_stats['new_B_grad_norm_mean']:.8f}")
    print(f"  frozen_old_A_max_abs_diff={max_a_diff:.10f}")
    print(f"  frozen_old_B_max_abs_diff={max_b_diff:.10f}")
    print(f"  cumulative_orth_check={cumulative_orth_formula_label(step_idx)}")


def run_rank_extension_variant(
    method_name,
    replay_per_class=0,
    use_orth=False,
    orth_mode=None,
    lambda_orth=0.0,
    zero_old_merge=False,
    orth_eval_records=None,
    orth_train_records=None,
    orth_summary_records=None,
):
    previous_rank_state = None
    stepwise_task_accuracies = {}
    active_orth_mode = None if orth_mode is None else str(orth_mode)
    active_lambda_orth = float(lambda_orth)

    for step_idx in range(NUM_STEPS):
        current_classes = classes_for_step(step_idx)
        trainable_classifier_classes = rank_extension_trainable_classifier_classes(
            step_idx=step_idx,
            replay_per_class=replay_per_class,
        )

        old_active_in_forward = not (zero_old_merge and step_idx > 0)
        model = build_rank_extension_model(
            previous_rank_state=previous_rank_state,
            step_idx=step_idx,
            old_active_in_forward=old_active_in_forward,
        )
        assert_rank_extension_structure(model=model, step_idx=step_idx)

        hooks = add_classifier_row_gradient_mask(
            model=model,
            trainable_classes=trainable_classifier_classes,
        )
        classifier_snapshot = snapshot_protected_classifier_rows(
            model=model,
            trainable_classes=trainable_classifier_classes,
        )
        frozen_snapshot = snapshot_frozen_rank_blocks(model)

        train_ds = make_train_dataset(step_idx=step_idx, replay_per_class=replay_per_class)
        eval_ds = make_eval_dataset(current_classes)
        out_dir = os.path.join(MODELS_DIR, f"{method_name}_step_{step_idx + 1}")

        trainer_cls = RankExtensionTrainer
        trainer_kwargs = {"classifier_snapshot": classifier_snapshot}
        if use_orth:
            trainer_cls = DeltaOrthRankExtensionTrainer
            trainer_kwargs.update({
                "lambda_orth": active_lambda_orth,
                "orth_mode": active_orth_mode,
                "orth_eps": float(ORTH_NORM_EPS),
                "log_every_steps": int(ORTH_LOSS_LOG_EVERY),
                "method_name": method_name,
                "step_idx": int(step_idx),
            })

        print(
            f"\n===== {method_name} | step {step_idx + 1}/{NUM_STEPS} | "
            f"rank={(step_idx + 1) * RANKEXT_NEW_RANK_PER_STEP} | "
            f"frozen_rank={0 if previous_rank_state is None else step_idx * RANKEXT_NEW_RANK_PER_STEP} | "
            f"new_rank={RANKEXT_NEW_RANK_PER_STEP} | "
            f"replay_per_class={replay_per_class} | "
            f"orth={use_orth} | "
            f"orth_mode={active_orth_mode} | "
            f"lambda_orth={active_lambda_orth:.6g} | "
            f"old_active_in_forward={old_active_in_forward} ====="
        )

        trainer, _ = train_with_trainer(
            model=model,
            train_ds=train_ds,
            eval_ds=eval_ds,
            output_dir=out_dir,
            epochs=RANKEXT_EPOCHS,
            lr=LR_RANKEXT,
            batch_size=BATCH_LORA,
            accum_steps=ACCUM_LORA,
            trainer_cls=trainer_cls,
            **trainer_kwargs,
        )

        if use_orth and isinstance(trainer, DeltaOrthRankExtensionTrainer):
            loss_rows_df = trainer.consume_logged_losses()
            if loss_rows_df is not None and orth_train_records is not None and len(loss_rows_df) > 0:
                orth_train_records.extend(loss_rows_df.to_dict("records"))

        frozen_diff_df = check_frozen_rank_blocks_unchanged(
            model=model,
            snapshot=frozen_snapshot,
            label=f"{method_name} step {step_idx + 1}",
            csv_path=os.path.join(
                TABLES_DIR,
                f"{method_name}_step_{step_idx + 1}_frozen_rank_blocks.csv",
            ),
        )
        restore_protected_classifier_rows(model, classifier_snapshot)
        protected_diff = classifier_protected_row_max_diff(model, classifier_snapshot)
        print(f"[rank_extension diagnostics] protected classifier row max diff after restore: {protected_diff:.10f}")

        grad_stats = collect_rank_block_grad_norms(model=model, train_ds=train_ds)
        print_rank_extension_step_diagnostics(
            method_name=method_name,
            step_idx=step_idx,
            model=model,
            replay_per_class=replay_per_class,
            old_active_in_forward=old_active_in_forward,
            frozen_diff_df=frozen_diff_df,
            grad_stats=grad_stats,
            eps=float(ORTH_NORM_EPS),
        )

        seen_acc = evaluate_seen_step_accuracies(model=model, upto_step_idx=step_idx)
        stepwise_task_accuracies[int(step_idx)] = seen_acc

        for h in hooks:
            h.remove()

        previous_rank_state = extract_rank_extension_state(model)
        del model
        cleanup()

    # Final evaluation always merges all learned deltas.
    final_rank_model = build_rank_extension_model(
        previous_rank_state=previous_rank_state,
        step_idx=NUM_STEPS - 1,
        old_active_in_forward=True,
    )
    eval_rows = evaluate_model(final_rank_model, method_name)

    avg_forgetting = compute_average_forgetting(stepwise_task_accuracies)
    print(f"[rank_extension summary] method={method_name} | avg_forgetting={avg_forgetting}")

    if use_orth and orth_eval_records is not None:
        for row in eval_rows:
            orth_eval_records.append({
                "method": row["method"],
                "eval_set": row["eval_set"],
                "accuracy": row["accuracy"],
                "loss": row["loss"],
                "orth_mode": active_orth_mode,
                "lambda_orth": float(active_lambda_orth),
                "zero_old_merge": bool(zero_old_merge),
                "avg_forgetting": float(avg_forgetting) if not np.isnan(avg_forgetting) else np.nan,
            })

    if orth_summary_records is not None:
        eval_map = {row["eval_set"]: float(row["accuracy"]) for row in eval_rows}
        orth_summary_records.append({
            "method": method_name,
            "orth_mode": "none" if not use_orth else active_orth_mode,
            "lambda_orth": float(active_lambda_orth),
            "zero_old_merge": bool(zero_old_merge),
            "first_step": eval_map.get("first_step", np.nan),
            "later_steps": eval_map.get("later_steps", np.nan),
            "all_seen": eval_map.get("all_seen", np.nan),
            "old_new_gap": eval_map.get("first_step", np.nan) - eval_map.get("later_steps", np.nan),
            "avg_forgetting": float(avg_forgetting) if not np.isnan(avg_forgetting) else np.nan,
        })

    del final_rank_model
    cleanup()


orth_sweep_eval_rows = []
orth_sweep_train_rows = []
orth_sweep_summary_rows = []

if METHODS_TO_RUN.get("rank_extension", False):
    run_rank_extension_variant(
        method_name="rank_extension",
        replay_per_class=0,
        use_orth=False,
        orth_summary_records=orth_sweep_summary_rows,
    )
else:
    print("Skipping rank_extension")

if METHODS_TO_RUN.get("rank_extension_orth_factor_lam_0p5", False):
    run_rank_extension_variant(
        method_name="rank_extension_orth_factor_lam_0p5",
        replay_per_class=0,
        use_orth=True,
        orth_mode="factor_orth",
        lambda_orth=0.5,
        zero_old_merge=False,
        orth_eval_records=orth_sweep_eval_rows,
        orth_train_records=orth_sweep_train_rows,
        orth_summary_records=orth_sweep_summary_rows,
    )
else:
    print("Skipping rank_extension_orth_factor_lam_0p5")

if METHODS_TO_RUN.get("rank_extension_orth_factor_lam_1", False):
    run_rank_extension_variant(
        method_name="rank_extension_orth_factor_lam_1",
        replay_per_class=0,
        use_orth=True,
        orth_mode="factor_orth",
        lambda_orth=1.0,
        zero_old_merge=False,
        orth_eval_records=orth_sweep_eval_rows,
        orth_train_records=orth_sweep_train_rows,
        orth_summary_records=orth_sweep_summary_rows,
    )
else:
    print("Skipping rank_extension_orth_factor_lam_1")

if METHODS_TO_RUN.get("rank_extension_orth_factor_lam_10", False):
    run_rank_extension_variant(
        method_name="rank_extension_orth_factor_lam_10",
        replay_per_class=0,
        use_orth=True,
        orth_mode="factor_orth",
        lambda_orth=10.0,
        zero_old_merge=False,
        orth_eval_records=orth_sweep_eval_rows,
        orth_train_records=orth_sweep_train_rows,
        orth_summary_records=orth_sweep_summary_rows,
    )
else:
    print("Skipping rank_extension_orth_factor_lam_10")

if METHODS_TO_RUN.get("rank_extension_orth_factor_lam_50", False):
    run_rank_extension_variant(
        method_name="rank_extension_orth_factor_lam_50",
        replay_per_class=0,
        use_orth=True,
        orth_mode="factor_orth",
        lambda_orth=50.0,
        zero_old_merge=False,
        orth_eval_records=orth_sweep_eval_rows,
        orth_train_records=orth_sweep_train_rows,
        orth_summary_records=orth_sweep_summary_rows,
    )
else:
    print("Skipping rank_extension_orth_factor_lam_50")

if len(orth_sweep_train_rows) > 0:
    orth_train_df = pd.DataFrame(orth_sweep_train_rows)
    orth_train_path = os.path.join(TABLES_DIR, "orth_lambda_sweep_train_losses.csv")
    orth_train_df.to_csv(orth_train_path, index=False)
    print("[rank_extension orth] saved training-loss diagnostics:", orth_train_path)

    factor_diag_df = (
        orth_train_df[orth_train_df["orth_mode"] == "factor_orth"]
        .groupby(["method", "lambda_orth"], as_index=False)
        .agg({
            "ce_loss": "mean",
            "factor_A_penalty_mean": "mean",
            "factor_B_penalty_mean": "mean",
            "factor_total_penalty_mean": "mean",
            "weighted_factor_orth_mean": "mean",
            "weighted_factor_orth_over_CE": "mean",
            "mean_A_overlap": "mean",
            "mean_B_overlap": "mean",
        })
        .rename(columns={"ce_loss": "ce_loss_mean"})
    )
    factor_diag_path = os.path.join(TABLES_DIR, "factor_orth_diagnostics_summary.csv")
    factor_diag_df.to_csv(factor_diag_path, index=False)
    print("[rank_extension factor_orth] saved diagnostics summary:", factor_diag_path)

if len(orth_sweep_summary_rows) > 0:
    orth_summary_df = pd.DataFrame(orth_sweep_summary_rows)
    orth_summary_path = os.path.join(TABLES_DIR, "orth_lambda_sweep_summary.csv")
    orth_summary_df.to_csv(orth_summary_path, index=False)
    print("[rank_extension orth] saved summary table:", orth_summary_path)
    display(orth_summary_df.sort_values("all_seen", ascending=False).round(6))


In [ ]:
# ============================================================
# 14) joint_upper_bound
# ============================================================

if METHODS_TO_RUN["joint_upper_bound"]:
    joint_model = fresh_pretrained_model()

    train_joint = make_joint_train_dataset()
    test_joint = make_joint_eval_dataset()

    print("\n===== joint_upper_bound =====")

    train_with_trainer(
        model=joint_model,
        train_ds=train_joint,
        eval_ds=test_joint,
        output_dir=os.path.join(MODELS_DIR, "joint_upper_bound"),
        epochs=JOINT_EPOCHS,
        lr=LR_JOINT,
        batch_size=BATCH_FT,
        accum_steps=ACCUM_FT,
    )

    evaluate_model(joint_model, "joint_upper_bound")

    del joint_model
    cleanup()

else:
    print("Skipping joint_upper_bound")

In [ ]:
# ============================================================
# 15) Final results table
# ============================================================

results_df = pd.DataFrame(all_results)

results_path = os.path.join(TABLES_DIR, "all_results_clip_vit_debug.csv")
results_df.to_csv(results_path, index=False)

print("Saved:", results_path)
results_df

In [ ]:
# ============================================================
# 18) Final pivot table + summary metrics
# ============================================================

def safe_lambda_tag_local(val):
    v = float(val)
    if abs(v - 1e-4) < 1e-12:
        return "lam_1em4"
    if abs(v - 1e-3) < 1e-12:
        return "lam_1em3"
    if abs(v - 0.01) < 1e-12:
        return "lam_001"
    if abs(v - 0.05) < 1e-12:
        return "lam_005"
    if abs(v - 0.1) < 1e-12:
        return "lam_01"
    s = f"{v:g}".replace("-", "m").replace(".", "p")
    return f"lam_{s}"


active_method_order = [
    "simple_avg",
    "rank_extension",
    "rank_extension_orth_factor_lam_0p5",
    "rank_extension_orth_factor_lam_1",
    "rank_extension_orth_factor_lam_10",
    "rank_extension_orth_factor_lam_50",
]

results_df = pd.DataFrame(all_results)
results_path = os.path.join(TABLES_DIR, "all_results_clip_vit_full_comparison.csv")
results_df.to_csv(results_path, index=False)
print("Saved raw results:", results_path)
display(results_df)

final_table = results_df.pivot_table(index="method", columns="eval_set", values="accuracy", aggfunc="mean")
for col in ["first_step", "later_steps", "all_seen"]:
    if col not in final_table.columns:
        final_table[col] = np.nan

final_table = final_table[["first_step", "later_steps", "all_seen"]]
present_order = [m for m in active_method_order if m in final_table.index]
extra_methods = [m for m in final_table.index if m not in active_method_order]
final_table = final_table.reindex(present_order + sorted(extra_methods))

final_table_percent = final_table * 100.0
summary_table = final_table_percent.copy()
summary_table["old_new_gap"] = summary_table["first_step"] - summary_table["later_steps"]

ranking_table = final_table_percent.sort_values("all_seen", ascending=False).copy()
ranking_table["rank_all_seen"] = np.arange(1, len(ranking_table) + 1)

final_table_path = os.path.join(TABLES_DIR, "final_accuracy_clip_vit_percent.csv")
summary_table_path = os.path.join(TABLES_DIR, "summary_metrics_clip_vit_percent.csv")
ranking_table_path = os.path.join(TABLES_DIR, "ranking_by_all_seen_percent.csv")

final_table_percent.to_csv(final_table_path)
summary_table.to_csv(summary_table_path)
ranking_table.to_csv(ranking_table_path)

print("\nSaved final accuracy table:", final_table_path)
print("Saved summary metrics table:", summary_table_path)
print("Saved ranking table:", ranking_table_path)

print("\nFinal accuracy (%):")
display(final_table_percent.round(2))

print("\nSummary metrics (%):")
display(summary_table.round(2))

print("\nRanking by all_seen (%):")
display(ranking_table.round(2))


In [ ]:
# ============================================================
# 19) Final grouped bar chart (active methods only)
# ============================================================

plot_df = final_table_percent.reset_index()

x = np.arange(len(plot_df))
width = 0.25

plt.figure(figsize=(14, 6))

plt.bar(x - width, plot_df["first_step"], width, label="first_step")
plt.bar(x, plot_df["later_steps"], width, label="later_steps")
plt.bar(x + width, plot_df["all_seen"], width, label="all_seen")

plt.xticks(x, plot_df["method"], rotation=30, ha="right")
plt.ylabel("Accuracy (%)")
plt.title("Final Accuracy Comparison (Active Methods)")
plt.legend()
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "01_grouped_accuracy_comparison.png")
plt.savefig(plot_path, dpi=200)
plt.show()
print("Saved:", plot_path)
